In [ ]:
import os 
root_dir="./lists"
for txt in os.listdir(root_dir):
    print(os.path.join(root_dir,txt))

In [ ]:
# ===== benchmark 测试集 (要保证 train 与它们相似度 <= 0.3) =====
L = "./lists"
benchmark_dict = {
    "protein": f"{L}/MaSIF_53.txt",      # 每行: 1EWY_A
    "DNA":     f"{L}/DNA-129_Test.txt",  # 3行fasta: >5dy0_A / 序列 / 标签
    "RNA":     f"{L}/RNA-117_Test.txt",  # 3行fasta
    "Ligand":  f"{L}/coach420.txt",      # 仅 PDB (无链名): 148L
    "ion":     f"{L}/metal_3d.txt",      # 仅 PDB (无链名)
    "hotspot": f"{L}/ppi-hotspot.txt",   # 每行: 1c2b-A
}
# 每个文件的解析方式: chain=每行pdb_chain / fasta=取>头 / dash=pdb-chain / pdb=只有pdb
KIND = {"protein": "chain", "DNA": "fasta", "RNA": "fasta",
        "Ligand": "pdb", "ion": "pdb", "hotspot": "dash"}
# 只给了 PDB(无链名)的集合 -> 用该 PDB 的所有链所在的簇 (保守, 宁可多排除)
PDBONLY = {"Ligand", "ion"}

# 30%/80% mmseqs 聚类文件: 同一簇内两条序列 >=30% identity, 作为"相似度>0.3"判据
cluster_txt = "./cluster/pdb_seqres_pro_clean_cluster_cluster_30_80_forSplit.txt"

# ===== 切分参数 =====
SEED = 42
OUT_DIR = "../final_split"
OUT_PREFIX = "cluster_30_80_benchclean"
TRAIN_VAL_RATIO = 8.0   # test 只装"污染簇"; 其余 free 簇按 train:val = 8:1 切

In [3]:
# ===== 1) 解析聚类文件 -> 数据池 (簇为单位) =====
import os, pickle, random
random.seed(SEED)

clusters = []      # cid -> [member 链 ...] (全小写 pdb_chain)
member2cid = {}    # member -> cid
pdb2cids = {}      # pdb -> set(cid)
with open(cluster_txt) as f:
    for line in f:
        toks = [t.strip().lower() for t in line.split() if t.strip()]
        if not toks:
            continue
        cid = len(clusters)
        clusters.append(toks)
        for t in toks:
            member2cid[t] = cid
            pdb2cids.setdefault(t.split("_")[0], set()).add(cid)

N = len(clusters)
TOTAL = sum(len(c) for c in clusters)
csize = lambda S: sum(len(clusters[c]) for c in S)   # 一组簇里的链(序列)总数
print(f"[pool] 簇 = {N}    链(序列) = {TOTAL}    覆盖 PDB = {len(pdb2cids)}")

[pool] 簇 = 49382    链(序列) = 962204    覆盖 PDB = 234329


In [4]:
# ===== 2) benchmark -> 污染簇 (含 benchmark 序列的簇, 禁止进 train) =====
def read_keys(path, kind):
    """返回该 benchmark 文件里的标识列表 (全小写)。"""
    out = []
    for line in open(path):
        s = line.strip()
        if not s:
            continue
        if kind == "fasta":
            if s.startswith(">"):
                out.append(s[1:].lower())          # >5dy0_A -> 5dy0_a
        elif kind == "dash":
            out.append(s.lower().replace("-", "_"))  # 1c2b-A -> 1c2b_a
        else:                                        # chain(MaSIF) 或 pdb
            out.append(s.lower())
    return out

forbidden = set()   # 污染簇 cid 集合
missing = []        # 既无链命中也无该 pdb 的 benchmark 标识
print("=== 每个 benchmark 贡献的污染簇 ===")
for name, path in benchmark_dict.items():
    kind = KIND[name]
    keys = set(read_keys(path, kind))
    before, n_miss = len(forbidden), 0
    for k in keys:
        if name in PDBONLY:                          # 只有 pdb -> 该 pdb 所有链的簇
            cids = pdb2cids.get(k, ())
            if not cids:
                n_miss += 1; missing.append((name, k)); continue
            forbidden.update(cids)
        elif k in member2cid:                        # 有链名且命中
            forbidden.add(member2cid[k])
        else:                                        # 链名没命中 -> 回退到该 pdb 所有链
            cids = pdb2cids.get(k.split("_")[0], ())
            if not cids:
                n_miss += 1; missing.append((name, k)); continue
            forbidden.update(cids)
    print(f"  {name:8s} keys={len(keys):4d}  本类命中后累计污染簇 +{len(forbidden)-before:<5d} 找不到 {n_miss}")

fb_m = csize(forbidden)
print(f"\n[forbidden] 污染簇 = {len(forbidden)} ({100*len(forbidden)/N:.2f}%)   含链 = {fb_m} ({100*fb_m/TOTAL:.2f}%)")
print(f"[forbidden] 完全找不到的标识 {len(missing)} 个 (不在聚类文件里, 池子内无对应序列): {missing[:10]}")

=== 每个 benchmark 贡献的污染簇 ===
  protein  keys=  53  本类命中后累计污染簇 +53    找不到 0
  DNA      keys= 129  本类命中后累计污染簇 +126   找不到 3
  RNA      keys= 116  本类命中后累计污染簇 +109   找不到 0
  Ligand   keys= 420  本类命中后累计污染簇 +387   找不到 2
  ion      keys=  59  本类命中后累计污染簇 +63    找不到 0
  hotspot  keys= 158  本类命中后累计污染簇 +146   找不到 0

[forbidden] 污染簇 = 884 (1.79%)   含链 = 119609 (12.43%)
[forbidden] 完全找不到的标识 5 个 (不在聚类文件里, 池子内无对应序列): [('DNA', '5udb_d'), ('DNA', '5udb_9'), ('DNA', '5udb_4'), ('Ligand', '2zcp'), ('Ligand', '1a2k')]


In [5]:
# ===== 3) 整簇切分: test=全部污染簇; free 簇按 train:val=8:1 (按链数贪心) =====
free = [c for c in range(N) if c not in forbidden]
random.shuffle(free)

tgt_train = csize(free) * TRAIN_VAL_RATIO / (TRAIN_VAL_RATIO + 1.0)

train, val, test = set(), set(), set(forbidden)
c_train = 0
for c in free:
    sz = len(clusters[c])
    if c_train + sz <= tgt_train or c_train == 0:
        train.add(c); c_train += sz
    else:
        val.add(c)

def report(name, S):
    m = csize(S)
    print(f"  {name:5s} 簇={len(S):6d}  链={m:7d}  ({100*m/TOTAL:.2f}%)")
print("[split] 按链(序列)数:")
report("train", train); report("val", val); report("test", test)

# 校验: train 不含污染簇; 三 split 互斥且覆盖全部簇
assert train.isdisjoint(forbidden), "train 含污染簇!"
assert train.isdisjoint(val) and train.isdisjoint(test) and val.isdisjoint(test)
assert len(train) + len(val) + len(test) == N
# 校验: 所有 benchmark 序列对应的簇都落在 test
for name, path in benchmark_dict.items():
    for k in set(read_keys(path, KIND[name])):
        cids = ([member2cid[k]] if (name not in PDBONLY and k in member2cid)
                else list(pdb2cids.get(k.split("_")[0] if name not in PDBONLY else k, ())))
        for c in cids:
            assert c in test, f"benchmark {name}:{k} 的簇没在 test!"
print("[check] train 无污染簇 ✓; 三 split 簇互斥且全覆盖 ✓; benchmark 全部落在 test ✓")

[split] 按链(序列)数:
  train 簇= 43158  链= 748973  (77.84%)
  val   簇=  5340  链=  93622  (9.73%)
  test  簇=   884  链= 119609  (12.43%)
[check] train 无污染簇 ✓; 三 split 簇互斥且全覆盖 ✓; benchmark 全部落在 test ✓


In [6]:
# ===== 4) 写出: 每个 split 的成员链表 + 簇分配/元信息 =====
# 注意: 聚类文件存在跨簇重复 token(同一 pdb_chain 出现在多个簇),
# 直接按簇写会让同一条链落入多个 split -> train/test 泄漏。
# 故按优先级 test > val > train 做 member 级去重, 保证三 split 的链串互斥。
os.makedirs(OUT_DIR, exist_ok=True)

seen = set()
split_members = {}
n_dup = 0
for name, S in [("test", test), ("val", val), ("train", train)]:   # 优先级顺序
    ms = []
    for c in sorted(S):
        for m in clusters[c]:
            if m in seen:
                n_dup += 1
                continue
            seen.add(m)
            ms.append(m)
    split_members[name] = ms

for name in ["train", "val", "test"]:
    path = os.path.join(OUT_DIR, f"{OUT_PREFIX}_{name}.txt")
    with open(path, "w") as fo:
        for m in split_members[name]:
            fo.write(m + "\n")     # 每行一个 pdb_chain (全小写)
    n = len(split_members[name])
    print(f"  写出 {name:5s}: {path}   ({n} 链, {100*n/sum(len(v) for v in split_members.values()):.2f}%)")
print(f"  [dedup] 跨 split 去重移除的重复链次数 = {n_dup}; 唯一链总数 = {len(seen)}")

# 簇 -> split 的分配, 以及污染/缺失信息, 存 pickle 便于复现与追溯
assign = {}
for c in train: assign[clusters[c][0]] = "train"
for c in val:   assign[clusters[c][0]] = "val"
for c in test:  assign[clusters[c][0]] = "test"
meta = {
    "seed": SEED,
    "train_val_ratio": TRAIN_VAL_RATIO,
    "n_clusters": N, "n_members_with_dup": TOTAL, "n_members_unique": len(seen),
    "counts": {k: len(v) for k, v in split_members.items()},
    "forbidden_cids": sorted(forbidden),
    "cluster_assignment": assign,          # 代表序列 -> split
    "missing_benchmark_keys": missing,     # 不在聚类文件里的 benchmark 标识
}
mp = os.path.join(OUT_DIR, f"{OUT_PREFIX}_meta.pkl")
with open(mp, "wb") as fo:
    pickle.dump(meta, fo)
print(f"  写出 meta : {mp}")

  写出 train: /home/lanxin/data/Interface_prediction/processing/data/datasets/cluster_30_80_benchclean_train.txt   (680105 链, 76.65%)
  写出 val  : /home/lanxin/data/Interface_prediction/processing/data/datasets/cluster_30_80_benchclean_val.txt   (89879 链, 10.13%)
  写出 test : /home/lanxin/data/Interface_prediction/processing/data/datasets/cluster_30_80_benchclean_test.txt   (117294 链, 13.22%)
  [dedup] 跨 split 去重移除的重复链次数 = 74926; 唯一链总数 = 887278
  写出 meta : /home/lanxin/data/Interface_prediction/processing/data/datasets/cluster_30_80_benchclean_meta.pkl
